# 한국어 챗봇 모델 학습 v2 (KoAlpaca-Polyglot-1.3B + LoRA)

이미 instruction 학습된 1.3B 한국어 모델을 베이스로, 강한 LoRA(r=16)로 추가 학습한다.
출발점이 좋아 답변 품질이 v1(polyglot 원본)보다 크게 향상된다.

런타임 → GPU(T4) 설정 후 실행.

In [ ]:
!pip -q install transformers datasets accelerate peft
!pip -q uninstall -y torchao

In [ ]:
import torch
print('GPU:', torch.cuda.is_available())

In [ ]:
from datasets import load_dataset
dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train').select(range(15000))
print(len(dataset), dataset[0])

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = 'EleutherAI/polyglot-ko-1.3b'
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
rt = tokenizer.decode(tokenizer.encode('파이썬이 뭐야?'), skip_special_tokens=True)
assert '파이썬' in rt.replace(' ',''), '토크나이저 이상'
print('토크나이저 OK:', rt)

In [ ]:
PROMPT = '### 질문: {q}\n\n### 답변: {a}'
MAX_LEN = 384
def fmt(ex):
    text = PROMPT.format(q=ex['instruction'].strip(), a=str(ex['output']).strip()) + tokenizer.eos_token
    return tokenizer(text, truncation=True, max_length=MAX_LEN)
tokenized = dataset.map(fmt, remove_columns=dataset.column_names)
print(tokenized)

In [ ]:
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
    target_modules=['query_key_value','dense','dense_h_to_4h','dense_4h_to_h'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
args = TrainingArguments(output_dir='out', per_device_train_batch_size=4,
    gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=2e-4,
    fp16=True, warmup_ratio=0.05, logging_steps=50, save_strategy='no', report_to='none')
Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator).train()

In [ ]:
merged = model.merge_and_unload()
SAVE='koalpaca-chatbot'
merged.save_pretrained(SAVE); tokenizer.save_pretrained(SAVE)
def ask(q,n=100):
    p='### 질문: '+q+'\n\n### 답변:'
    ids=tokenizer.encode(p,return_tensors='pt').to(merged.device)
    o=merged.generate(ids,max_new_tokens=n,do_sample=True,top_p=0.92,temperature=0.7,
        repetition_penalty=1.1,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(o[0][ids.shape[1]:],skip_special_tokens=True).split('###')[0].strip()
for q in ['파이썬이 뭐야?','건강하게 사는 방법 알려줘','딥러닝과 머신러닝의 차이는?']:
    print('Q:',q); print('A:',ask(q)); print()

In [ ]:
from huggingface_hub import login, create_repo
login(token='hf_본인_WRITE_토큰')
REPO='jjminu/koalpaca-chatbot'
create_repo(REPO, exist_ok=True)
merged.push_to_hub(REPO); tokenizer.push_to_hub(REPO)
print('업로드 완료:', REPO)